# Module 3 — Retrieval

Part of the RAG Playbook. Program and progress: [CURRICULUM.md](CURRICULUM.md).

## Why you're here: what this module is for

**Where this module starts**
- RAG (retrieval-augmented generation) answers a question in two steps:
  1. A search program finds the few texts in a collection that match the question.
  2. A language model, the kind of program behind chat assistants, reads those texts and writes the answer.
- An **embedding** is a list of numbers that an embedding model makes from a text. Texts with similar meaning get embeddings that point in similar directions.
- **Cosine similarity** is one number, from −1 to 1, for how close the directions of two embeddings are: 1 means the same direction.
- An embedding follows the topic of a text more than its exact words: two sentences that differ by one name or one number can get almost the same embedding.

**This module's point:** A RAG system is built from five parts:
1. Chunk: split each document into short pieces
2. Embed: turn each piece into an embedding
3. **Store**: keep every piece's embedding in one table
4. **Retrieve**: for each question, return the pieces that match it best
5. Generate: the language model reads those pieces and writes the answer

it builds the store and the retrieval, and compares three ways to retrieve: by meaning, by words, and by both combined.<br>
<u>Note</u>: this module skips part 1, chunk: its texts are single sentences, already short. Part 2, embed, uses an existing embedding model, nomic-embed-text, as its makers trained it. Part 5, generate, is not run: this module stops at the sentences the search returns.

**What this module tests**

The collection is sentences from news articles, taken from a public dataset called MultiHop-RAG. The dataset has questions about the articles, and for each question it lists the sentences in the articles that support its answer. So for a question, a good search returns the sentences listed for it.

The three ways to retrieve:
1. Meaning search: return the sentences whose embeddings are closest to the question's embedding.
2. Keyword search: return the sentences that share the most words with the question, with rare words counting more than common ones.
3. Hybrid search: combine the two rankings into one.

The module runs five steps:
1. Store: turn every sentence into an embedding, and keep the embeddings in one table.
2. Meaning search on one question: which sentences does it return, and are they the ones listed for the question?
3. Keyword search on the same question: which sentences does it return?
4. Hybrid search on the same question: which sentences does it return?
5. Two more questions, searched the three ways side by side: where does each way find the listed sentences, and where does it miss them?

**Two tags in this module**
- **[in real life]** marks a step a real search system runs.
- **[only in this module]** marks a step that uses the dataset's list of supporting sentences. In real life, when a user asks a new question, nobody has that list: finding those sentences is the search's job.

**What this module produces**
1. A store: every sentence's embedding, kept in one table.
2. Three ways to retrieve: meaning search, keyword search and hybrid search, with keyword search's and hybrid search's scores worked by hand on small examples.
3. A side-by-side of the three ways on three questions, showing where each one finds and misses the listed sentences.

## From the curriculum

> ## <u>Module 3 — Retrieval</u>
>
> **Concepts**
> - The vector store: every chunk's embedding, kept in one table to search
> - Top-k search: the k chunks closest to the question
> - Keyword search (BM25, Best Matching 25: a word-matching score) vs. meaning search (embeddings)
> - Hybrid search: combining the two rankings
>
> **Worked example:** one question searched three ways, side by side.

## Lesson

In this module every piece, called a chunk in the curriculum above, is one sentence.

### The store and top-k search

- The store is a table with one row per sentence: the sentence's text and its embedding. In RAG writing it is called a **vector store**: "vector" is the mathematical name for a list of numbers, here the embedding.
- Every sentence is turned into an embedding once, when the collection is stored. A new question is turned into an embedding when it is asked.
- **Top-k search** returns the k sentences that score highest for the question, where k is a number chosen in advance, such as 4. Meaning search scores each sentence by the cosine similarity of its embedding with the question's embedding.

### Keyword search: BM25

Meaning search can rank low a sentence that shares the question's exact words, such as a name or a date. An embedding follows the topic more than the exact words, so a shared name moves two embeddings only a little closer, and sentences on the same topic without that name can score as high or higher. Keyword search scores by the words themselves.

**BM25** (Best Matching 25) is the standard keyword score. It scores each sentence for a question with three ideas:
- Every word of the question that also appears in the sentence adds to the sentence's score.
- A rare word adds more than a common one. A person's surname appears in few sentences, so a sentence that has it is likely about the question's topic. "The" appears in most sentences, so it says almost nothing.
- A word adds more when it appears more often in the sentence, but each repeat adds less than the one before. A long sentence gets less for the same word, because a long sentence contains many words, so it is more likely to contain any given word by chance.

A word here is a run of letters and digits, in lowercase: "Wi-Fi" is the two words "wi" and "fi". Each different word of the question counts once.

**formula**:

BM25 score of a sentence = $\sum_{\text{words of the question}}$ (rarity weight of the word) × (count part of the word), where the sum adds over every different word of the question

Rarity weight of a word = $\ln\left(1 + \frac{(\text{sentences in the collection}) - (\text{sentences with the word}) + 0.5}{(\text{sentences with the word}) + 0.5}\right)$

Count part of a word = $\frac{(\text{times in the sentence}) \times (1.2 + 1)}{(\text{times in the sentence}) + 1.2 \times \left(1 - 0.75 + 0.75 \times \frac{\text{words in the sentence}}{\text{average words per sentence}}\right)}$<br>
<u>Note</u>: 1.2 and 0.75 are settings chosen in advance, not numbers calculated from the data.
- 1.2: any number from 0 up. At 0, repeats add nothing: a word scores the same once or five times. Common values run from 1.2 to 2.0.
- 0.75: any number from 0 to 1. At 0, sentence length does not count, and at 1 it counts fully. 0.75 is the common choice.

In words:
- The rarity weight is large when few sentences have the word, and close to 0 when most sentences have it. ln is the natural logarithm: it grows ever more slowly, so the weights of a rare and a common word differ far less than how often the two words appear.
- The fraction inside ln( ) is below 1 when more than half the sentences have the word, and ln of a number between 0 and 1 is negative. A weight below 0 would lower the score of a sentence for containing the word. Adding 1 to the fraction keeps every rarity weight above 0.
- The two 0.5s are part of the standard formula: the one below the line keeps the fraction from dividing by 0 for a word that no sentence has. This module skips such words anyway, because they add to no sentence's score.
- The count part is 0 when the word is not in the sentence, so only shared words add to the score. It is 1 when the word appears once in a sentence of average length.
- 1.2 sets the most that repeats can add. When a word repeats many times, (times in the sentence) outweighs everything else in the bottom of the fraction, so the count part approaches (times in the sentence × 2.2) / (times in the sentence) = 2.2, and never goes above it.
- The bracket (1 − 0.75 + 0.75 × words in the sentence / average words per sentence) is 1 for a sentence of average length, above 1 for a longer sentence and below 1 for a shorter one. A larger bracket makes the count part smaller. 0.75 sets how strongly the length moves the bracket: with 0 instead, length would not count at all.
- 1.2 and 0.75 are the standard values, the defaults in Lucene, the search library behind many website search boxes.

A hand example, in a collection of 1,000 sentences, chosen for easy arithmetic. In each count part below, the number after "0.75 ×" is (words in the sentence) / (average words per sentence): 1 for a sentence of average length, 2 for a sentence twice that length.

- Rarity weight of a word in 2 sentences = ln(1 + ((1,000 − 2 + 0.5) / (2 + 0.5))) = ln(1 + 399.4) = ln(400.4) = 5.99
- Rarity weight of a word in 500 sentences = ln(1 + ((1,000 − 500 + 0.5) / (500 + 0.5))) = ln(1 + 1) = ln(2) = 0.69

<!-- -->

- Count part of a word once, in a sentence of average length = (1 × 2.2) / (1 + (1.2 × (1 − 0.75 + (0.75 × 1)))) = 2.2 / (1 + (1.2 × 1)) = 2.2 / 2.2 = 1
- Count part of a word twice, in a sentence of average length = (2 × 2.2) / (2 + (1.2 × (1 − 0.75 + (0.75 × 1)))) = 4.4 / (2 + (1.2 × 1)) = 4.4 / 3.2 = 1.38
- Count part of a word once, in a sentence twice the average length = (1 × 2.2) / (1 + (1.2 × (1 − 0.75 + (0.75 × 2)))) = 2.2 / (1 + (1.2 × 1.75)) = 2.2 / 3.1 = 0.71

So the rare word, found once in a sentence of average length, adds 5.99 × 1 = 5.99 to the sentence's score. The common word adds 0.69 × 1 = 0.69. The rare word's weight is about (5.99 / 0.69 =) 9 times the common word's, though the common word is in 500 / 2 = 250 times as many sentences. Appearing twice raises the count part from 1 to 1.38, not to 2. A sentence twice the average length lowers it from 1 to 0.71.

### Hybrid search: reciprocal rank fusion

Meaning search and keyword search each rank all the sentences, from 1 (best) down. Hybrid search combines the two rankings into one.

The two scores cannot simply be added: a cosine similarity is at most 1, while a BM25 score grows with every word the sentence shares with the question (one rare word alone added 5.99 in the example above), so the BM25 score would decide almost alone. The ranks are on the same scale for both, so hybrid search combines the ranks.

**Reciprocal rank fusion** is the standard way to combine them ("reciprocal" means 1 divided by the number):

**formula**:

Hybrid score of a sentence = $\frac{1}{60 + (\text{rank in meaning search})} + \frac{1}{60 + (\text{rank in keyword search})}$

In words:
- A sentence ranked near the top by either search gets a larger hybrid score. The sentences are then ranked by hybrid score, and the top k are returned.
- 60 is the value the method's authors (Cormack, Clarke and Büttcher, 2009) found to work well. Because 60 is added to every rank, 1 / (60 + rank) changes little from one rank to the next: rank 1 gives 1 / 61 = 0.01639, rank 5 gives 1 / 65 = 0.01538. So first place in one search is worth little more than fifth place, and a sentence ranked high by both searches can beat one ranked first by one search and lower by the other, as in the example below.

A hand example with two sentences, P and Q, ranks chosen for easy arithmetic:

- Hybrid score of P, ranked 1 by meaning and 30 by keywords = (1 / (60 + 1)) + (1 / (60 + 30)) = 0.01639 + 0.01111 = 0.02750
- Hybrid score of Q, ranked 5 by meaning and 5 by keywords = (1 / (60 + 5)) + (1 / (60 + 5)) = 0.01538 + 0.01538 = 0.03077

So Q, ranked 5th by both searches, comes before P. Without the 60, P would score (1 / 1) + (1 / 30) = 1.03 and Q (1 / 5) + (1 / 5) = 0.40, and P's first place would decide alone.

The code below loads the tools and checks the BM25 and hybrid functions used in this module against the hand examples above: each printed number should equal its hand calculation. The sentence lengths in the code, 20 and 40 words against an average of 20, give a sentence of average length and one twice the average. `common` is this playbook's shared helper file: it loads the data, calls the models, and prints tables.

In [1]:
import re
from collections import Counter

import numpy as np
import pandas as pd

import common
from common import show

K1, B = 1.2, 0.75   # BM25's standard values
RRF = 60            # reciprocal rank fusion's standard value
TOP_K = 4


def words_of(text):
    return re.findall(r"[a-z0-9]+", text.lower())


def rarity_weight(sentences_with_word, sentences_in_collection):
    return np.log(1 + (sentences_in_collection - sentences_with_word + 0.5) / (sentences_with_word + 0.5))


def count_part(times_in_sentence, words_in_sentence, average_words):
    return (times_in_sentence * (K1 + 1)) / (times_in_sentence + K1 * (1 - B + B * words_in_sentence / average_words))


def hybrid_score(meaning_rank, keyword_rank):
    return 1 / (RRF + meaning_rank) + 1 / (RRF + keyword_rank)


print(f"Rarity weight of a word in 2 sentences of 1,000 = {rarity_weight(2, 1000):.2f}")
print(f"Rarity weight of a word in 500 sentences of 1,000 = {rarity_weight(500, 1000):.2f}")
print()
print(f"Count part of a word once, average length = {count_part(1, 20, 20):.2f}")
print(f"Count part of a word twice, average length = {count_part(2, 20, 20):.2f}")
print(f"Count part of a word once, twice the average length = {count_part(1, 40, 20):.2f}")
print()
print(f"Hybrid score of P (ranks 1 and 30) = {hybrid_score(1, 30):.5f}")
print(f"Hybrid score of Q (ranks 5 and 5) = {hybrid_score(5, 5):.5f}")

Rarity weight of a word in 2 sentences of 1,000 = 5.99
Rarity weight of a word in 500 sentences of 1,000 = 0.69

Count part of a word once, average length = 1.00
Count part of a word twice, average length = 1.38
Count part of a word once, twice the average length = 0.71

Hybrid score of P (ranks 1 and 30) = 0.02750
Hybrid score of Q (ranks 5 and 5) = 0.03077


## The data: the sentences the questions list

The dataset is MultiHop-RAG (Tang and Yang, 2024):
- 609 news articles, published between September and December 2023
- 2,556 questions about those articles. For 2,255 of them, the dataset lists 2 to 4 sentences from the articles that support the answer. The other 301 ask something no article answers, and list no sentences.

**[only in this module]** The collection searched in this module is every sentence the dataset lists, for any question. In real life, the collection would be every sentence of all 609 articles, most of which no question lists, so finding the listed ones would be harder. Splitting the articles into sentences is part 1, chunk, which this module skips.

The dataset reuses many sentences across questions, so each unique sentence is stored once. For any one question, most of the collection's sentences are listed for other questions, not for it, so each search has to pick the 2 to 4 listed ones out of the whole collection.

Each search returns its top 4 sentences: k = 4, because no question lists more than 4 sentences (printed below), so 4 leaves room for every listed sentence of any question.

In [2]:
questions = common.load_questions()
print(f"Articles = {len(common.load_corpus()):,}")
print(f"Questions = {len(questions):,}")
print(f"Questions with listed sentences = {questions['evidence_list'].str.len().gt(0).sum():,}")
print(f"Questions with no listed sentences = {questions['evidence_list'].str.len().eq(0).sum():,}")
print()

rows = [(q, e["fact"]) for q, listed in questions["evidence_list"].items() for e in listed]
listed = pd.DataFrame(rows, columns=["question", "sentence"])
sentences = listed.drop_duplicates("sentence").reset_index(drop=True)
row_of = {s: i for i, s in enumerate(sentences["sentence"])}

print(f"Sentences listed, over all questions = {len(listed):,}")
print(f"Unique sentences in the collection = {len(sentences):,}")
print(f"Times each unique sentence is listed, on average = {len(listed):,} / {len(sentences):,} = {len(listed) / len(sentences):.1f}")
print()

per_question = listed.groupby("question").size().value_counts().sort_index()
table = pd.DataFrame({"sentences listed": [str(n) for n in per_question.index] + ["total"],
                      "questions": [f"{v:,}" for v in per_question] + [f"{per_question.sum():,}"],
                      "share": [f"{v / per_question.sum():.1%}" for v in per_question] + ["100%"]})
print("Questions with listed sentences, by number of sentences listed:")
print()
show(table)

Articles = 609
Questions = 2,556
Questions with listed sentences = 2,255
Questions with no listed sentences = 301

Sentences listed, over all questions = 6,084
Unique sentences in the collection = 981
Times each unique sentence is listed, on average = 6,084 / 981 = 6.2

Questions with listed sentences, by number of sentences listed:

sentences listed  questions  share
2                     1,079  47.8%
3                       778  34.5%
4                       398  17.6%
total                 2,255   100%


## Step 1 of 5: store every sentence's embedding

**[in real life]** Every sentence in the collection is turned into an embedding once, and the embeddings are kept in one table, the vector store, with one row per sentence.

The embedding model, nomic-embed-text, was trained with a short label in front of every text, saying what the text is for: "search_document: " in front of a passage to be searched, and "search_query: " in front of a question. Its makers ask for the same labels when it is used, so each sentence gets "search_document: " here, and each question gets "search_query: " when it is searched.

In [3]:
store = common.embed(sentences["sentence"].tolist(), prefix="search_document: ")
print(f"Vector store = {store.shape[0]:,} rows (sentences) × {store.shape[1]} numbers per embedding")

Vector store = 981 rows (sentences) × 768 numbers per embedding


## Step 2 of 5: meaning search on one question

**Which sentences does meaning search return for one question, and are they the ones listed for it?** The question below was chosen because the three ways of searching return different sentences for it, which shows what each way does. It asks about two topics at once: a streamer's view of a platform, and a change in how that platform shares its income.

**[in real life]** The question is turned into an embedding, every sentence in the vector store gets its cosine similarity with that embedding, and the 4 sentences with the highest cosine similarity are returned.

**[only in this module]** Each returned sentence is checked against the sentences the dataset lists for the question, labelled L1, L2 and so on. The output also gives the rank, of all the sentences, that meaning search gave each listed sentence.

In [4]:
def meaning_scores(question):
    q = common.embed([question], prefix="search_query: ")[0]
    return pd.Series(store @ q / (np.linalg.norm(store, axis=1) * np.linalg.norm(q)))


def ranks_of(scores):
    # Rank 1 = highest score. Ties (sentences with the same score) keep the collection's order.
    return scores.rank(ascending=False, method="first").astype(int)


def listed_for(q):
    return [row_of[s] for s in dict.fromkeys(listed.loc[listed["question"] == q, "sentence"])]


def print_question(q):
    print(f"Question: {questions['query'][q]}")
    print(f"Answer in the dataset: {questions['answer'][q]}")
    print()
    print("Sentences listed for this question:")
    for k, i in enumerate(listed_for(q), start=1):
        print(f"  L{k}: {sentences['sentence'][i]}")


def print_top(q, scores, score_name, decimals):
    gold = listed_for(q)
    ranks = ranks_of(scores)
    print(f"Top {TOP_K} sentences:")
    print()
    for i in ranks.sort_values().index[:TOP_K]:
        mark = f"listed sentence L{gold.index(i) + 1}" if i in gold else "not listed"
        print(f"Rank {ranks[i]}, {score_name} = {scores[i]:.{decimals}f}, {mark}")
        print(f"  {sentences['sentence'][i]}")
    print()
    print(f"Rank of each listed sentence, of all {len(sentences):,}:")
    for k, i in enumerate(gold, start=1):
        print(f"  L{k} = rank {ranks[i]}")


Q_MAIN = 1716
print_question(Q_MAIN)
print()
meaning = meaning_scores(questions["query"][Q_MAIN])
print_top(Q_MAIN, meaning, "cosine similarity", 3)

Question: Which platform is favored by Keira Megan for fan engagement and authenticity, and has also been in the news on TechCrunch for altering its revenue split to focus more on ad revenue?
Answer in the dataset: Twitch

Sentences listed for this question:
  L1: Megan says that while she doesn’t have as many followers on Twitch as she does on her YouTube channel, there are elements of Twitch she prefers to YouTube, including the ability to be herself and engage with fans more closely.
  L2: Right, and that decision to change the revenue split also came out of a decision to prioritize ad revenue.

Top 4 sentences:

Rank 1, cosine similarity = 0.625, listed sentence L1
  Megan says that while she doesn’t have as many followers on Twitch as she does on her YouTube channel, there are elements of Twitch she prefers to YouTube, including the ability to be herself and engage with fans more closely.
Rank 2, cosine similarity = 0.606, not listed
  Users today have far more options for microbl

**Observation:** meaning search returned listed sentence L1 first: L1 is about the streamer, Megan, and her view of Twitch, and it shares the question's topic of fans and a platform, though it says "engage with fans" where the question says "fan engagement".

**Observation:** meaning search ranked L2, about the revenue split, 28th, below 27 sentences that are not listed.

**Observation:** the sentence at rank 3 is not listed, but it describes the same kind of decision as L2: ending a revenue split deal in favor of ad revenue. Not listed does not mean unrelated: the dataset lists the sentences its authors chose for the question, and other sentences may be about the same topic.

**Conclusion:** meaning search found one of the two listed sentences and missed the other, though it returned an unlisted sentence on the other's topic.

## Step 3 of 5: keyword search on the same question

**Does keyword search find L2, the sentence meaning search missed?** L2 shares the question's exact words "revenue split" and "ad revenue", and keyword search scores exact words.

**[in real life]** Every sentence in the collection gets its BM25 score for the question, and the 4 sentences with the highest score are returned. The sentences' word counts are taken once, when the collection is stored, like the embeddings.

**[only in this module]** Each returned sentence is checked against the listed sentences, as in Step 2.

First, the rarity weight of each word of the question. The table below shows which of the question's words can lift a sentence's score most: it lists each different word, how many of the collection's sentences have it, and its rarity weight, from the rarest word down. A word in no sentence can add to no sentence's score, so it gets no rarity weight, shown as "—" at the bottom.

In [5]:
sentence_words = [words_of(s) for s in sentences["sentence"]]
word_counts = [Counter(w) for w in sentence_words]
words_per_sentence = np.array([len(w) for w in sentence_words])
average_words = words_per_sentence.mean()
sentences_with = Counter(w for ws in sentence_words for w in set(ws))
N = len(sentences)


def keyword_scores(question):
    total = np.zeros(N)
    for w in dict.fromkeys(words_of(question)):
        if sentences_with[w] == 0:
            continue
        times = np.array([c[w] for c in word_counts])
        total += rarity_weight(sentences_with[w], N) * count_part(times, words_per_sentence, average_words)
    return pd.Series(total)


question_words = list(dict.fromkeys(words_of(questions["query"][Q_MAIN])))
table = pd.DataFrame({"word": question_words,
                      "sentences with the word": [sentences_with[w] for w in question_words]})
table["rarity weight"] = [rarity_weight(n, N) if n else -1.0 for n in table["sentences with the word"]]
table = table.sort_values("rarity weight", ascending=False)
table["rarity weight"] = table["rarity weight"].map(lambda v: f"{v:.2f}" if v >= 0 else "—")
print(f"Sentences in the collection = {N:,}")
print()
print("Words of the question, rarest first:")
print()
show(table)

Sentences in the collection = 981

Words of the question, rarest first:

word          sentences with the word  rarity weight
favored                             1           6.48
megan                               1           6.48
techcrunch                          1           6.48
engagement                          2           5.97
split                               3           5.64
ad                                  4           5.39
fan                                 4           5.39
focus                               6           5.02
platform                           14           4.22
revenue                            14           4.22
news                               15           4.15
more                               61           2.77
which                              63           2.74
its                                69           2.65
been                               72           2.61
also                               75           2.57
by                        

"Keira", "authenticity" and "altering" appear in no sentence, so they add nothing to any sentence's score. "The", "and" and "to" each appear in more than half the 981 sentences, so their rarity weights are below 0.69, the weight of a word in exactly half.

Now the search itself:

In [6]:
keyword = keyword_scores(questions["query"][Q_MAIN])
print_top(Q_MAIN, keyword, "BM25 score", 2)

Top 4 sentences:

Rank 1, BM25 score = 23.87, listed sentence L2
  Right, and that decision to change the revenue split also came out of a decision to prioritize ad revenue.
Rank 2, BM25 score = 17.92, not listed
  Last year, the company announced that it would end the 70/30 subscription revenue split deal that it had offered select streamers in favor of prioritizing ad revenue.
Rank 3, BM25 score = 17.28, not listed
  The case, filed by Arkansas-based publisher Helena World Chronicle, argues that Google “siphons off” news publishers’ content, their readers and ad revenue through anticompetitive means.
Rank 4, BM25 score = 14.79, not listed
  Since its first-quarter launch, it’s seen its revenue grow by more than 50% quarter over quarter.

Rank of each listed sentence, of all 981:
  L1 = rank 25
  L2 = rank 1


**Observation:** keyword search returned L2 first, the sentence meaning search ranked 28th, and ranked L1, which meaning search returned first, 25th.

To see why, the output below breaks each listed sentence's BM25 score into the words it shares with the question, the largest addition first.

In [7]:
def print_breakdown(q, i, label):
    shared = [w for w in dict.fromkeys(words_of(questions["query"][q])) if word_counts[i][w] > 0]
    rows = []
    for w in shared:
        weight = rarity_weight(sentences_with[w], N)
        part = count_part(word_counts[i][w], words_per_sentence[i], average_words)
        rows.append([w, sentences_with[w], f"{weight:.2f}", word_counts[i][w], f"{part:.2f}", f"{weight * part:.2f}"])
    rows.sort(key=lambda r: -float(r[5]))
    table = pd.DataFrame(rows, columns=["word", "sentences with it", "rarity weight", "times in sentence",
                                        "count part", "rarity weight × count part"])
    print(f"{label}, {words_per_sentence[i]} words:")
    print()
    show(table)
    rounded_sum = sum(float(v) for v in table["rarity weight × count part"])
    print()
    print(f"BM25 score of {label} = sum of the last column = {keyword[i]:.2f}"
          f" (adding the rounded numbers above gives {rounded_sum:.2f})")


gold = listed_for(Q_MAIN)
print(f"Average words per sentence = {average_words:.1f}")
print()
print_breakdown(Q_MAIN, gold[1], "L2")
print()
print_breakdown(Q_MAIN, gold[0], "L1")

Average words per sentence = 27.5

L2, 19 words:

word     sentences with it  rarity weight  times in sentence  count part  rarity weight × count part
split                    3           5.64                  1        1.14                        6.45
revenue                 14           4.22                  2        1.51                        6.35
ad                       4           5.39                  1        1.14                        6.16
also                    75           2.57                  1        1.14                        2.94
to                     519           0.64                  2        1.51                        0.96
and                    513           0.65                  1        1.14                        0.74
the                    769           0.24                  1        1.14                        0.28

BM25 score of L2 = sum of the last column = 23.87 (adding the rounded numbers above gives 23.88)

L1, 41 words:

word   sentences with it  ra

**Observation:** L2's score comes mostly from "split", "revenue" and "ad", three rare words. L2 has 19 words, fewer than the average of 27.5, so a word found once in L2 gets a count part of 1.14, above 1. "Revenue" appears twice, so its count part is 1.51.

**Observation:** L1 shares fewer rare words with the question. "Megan" is rare and adds the most. But L1 says "fans" and "engage" where the question says "fan" and "engagement": keyword search counts only identical words, so those add nothing. L1 has 41 words, more than the average of 27.5, so a word found once in L1 gets a count part of 0.83, below 1.

**Conclusion:** keyword search found L2, the sentence that repeats the question's words, and missed L1, the sentence that matches the question's meaning but uses other forms of its words.

## Step 4 of 5: hybrid search on the same question

**Does combining the two rankings return both listed sentences?** Meaning search ranked L1 first and L2 28th. Keyword search ranked L2 first and L1 25th.

**[in real life]** Every sentence gets its hybrid score from its two ranks, and the 4 sentences with the highest hybrid score are returned.

**[only in this module]** Each returned sentence is checked against the listed sentences, as in Step 2.

In [8]:
meaning_rank, keyword_rank = ranks_of(meaning), ranks_of(keyword)
hybrid = pd.Series(hybrid_score(meaning_rank.values, keyword_rank.values))
print_top(Q_MAIN, hybrid, "hybrid score", 5)
print()
print("Hybrid score = 1 / (60 + rank in meaning search) + 1 / (60 + rank in keyword search)")
for i in ranks_of(hybrid).sort_values().index[:TOP_K]:
    print(f"  Rank {ranks_of(hybrid)[i]}, hybrid score = 1 / (60 + {meaning_rank[i]}) + 1 / (60 + {keyword_rank[i]})"
          f" = {1 / (RRF + meaning_rank[i]):.5f} + {1 / (RRF + keyword_rank[i]):.5f} = {hybrid[i]:.5f}")

Top 4 sentences:

Rank 1, hybrid score = 0.03200, not listed
  Last year, the company announced that it would end the 70/30 subscription revenue split deal that it had offered select streamers in favor of prioritizing ad revenue.
Rank 2, hybrid score = 0.02816, listed sentence L1
  Megan says that while she doesn’t have as many followers on Twitch as she does on her YouTube channel, there are elements of Twitch she prefers to YouTube, including the ability to be herself and engage with fans more closely.
Rank 3, hybrid score = 0.02776, listed sentence L2
  Right, and that decision to change the revenue split also came out of a decision to prioritize ad revenue.
Rank 4, hybrid score = 0.02737, not listed
  The case, filed by Arkansas-based publisher Helena World Chronicle, argues that Google “siphons off” news publishers’ content, their readers and ad revenue through anticompetitive means.

Rank of each listed sentence, of all 981:
  L1 = rank 2
  L2 = rank 3

Hybrid score = 1 / (60 + r

**Observation:** hybrid search returned both listed sentences, L1 and L2, in its top 4, where meaning search and keyword search each returned one.

**Observation:** the sentence at rank 1 is ranked 3 in meaning search, about ending a revenue split deal: it is not listed, but both searches ranked it near the top, 3rd and 2nd, and a sentence ranked high by both can beat one ranked first by one search alone.

**Conclusion:** for this question, combining the two rankings returned both listed sentences, because each search found the one the other missed.

## Step 5 of 5: two more questions, the three ways side by side

**Is one way always better?** Two more questions, chosen because they show the opposite cases: in the first, keyword search finds more of the listed sentences than meaning search; in the second, meaning search finds more. Two hand-picked questions show what can happen, not how often it happens.

**[in real life]** Each question is searched the three ways, as an analyst does on test questions with known supporting sentences, to choose the one method the live search will use.

**[only in this module]** For each listed sentence, the output gives its rank in each of the three searches, then counts how many listed sentences each search returned in its top 4.

<u>Question 1</u>

In [9]:
def side_by_side(q):
    print_question(q)
    print()
    m = ranks_of(meaning_scores(questions["query"][q]))
    k = ranks_of(keyword_scores(questions["query"][q]))
    h = ranks_of(pd.Series(hybrid_score(m.values, k.values)))
    gold = listed_for(q)
    table = pd.DataFrame({"listed sentence": [f"L{n}" for n in range(1, len(gold) + 1)],
                          "meaning rank": [m[i] for i in gold],
                          "keyword rank": [k[i] for i in gold],
                          "hybrid rank": [h[i] for i in gold]})
    print(f"Rank of each listed sentence, of all {len(sentences):,}:")
    print()
    show(table)
    print()
    print(f"Listed sentences in the top {TOP_K}:")
    for name, r in [("Meaning search", m), ("Keyword search", k), ("Hybrid search", h)]:
        print(f"  {name} = {sum(r[i] <= TOP_K for i in gold)} of {len(gold)}")
    hybrid = hybrid_score(m, k)
    for name, r in [("Meaning search", m), ("Keyword search", k), ("Hybrid search", h)]:
        print()
        print(f"{name}'s top {TOP_K}, with each sentence's meaning rank, keyword rank and hybrid score:")
        for i in r.sort_values().index[:TOP_K]:
            mark = f"listed sentence L{gold.index(i) + 1}" if i in gold else "not listed"
            print(f"  Rank {r[i]}, meaning rank {m[i]}, keyword rank {k[i]}, hybrid score {hybrid[i]:.5f}, {mark}")
            print(f"    {sentences['sentence'][i]}")


side_by_side(2125)

Question: Which company, covered by Engadget and Polygon, is releasing a new version of their product on November 16th, featuring several minor physical upgrades and an emphasis on a unified performance target for developers?
Answer in the dataset: Valve

Sentences listed for this question:
  L1: Valve has announced a new Steam Deck and — double surprise — we’ve already reviewed it.
  L2: And so, alongside the big improvements, Valve has made a gaggle of minor physical upgrades with its new iteration.
  L3: For now, Griffais explained, it's important to Valve that developers don't need to "split their attention" between different performance targets.
  L4: The Steam Deck OLED will go on sale on November 16th at 1pm ET, and Valve has units ready to ship that same day.

Rank of each listed sentence, of all 981:

listed sentence  meaning rank  keyword rank  hybrid rank
L1                        121           253          162
L2                          9             1            2
L3     

**Observation:** keyword search returned 2 of the 4 listed sentences, L2 and L3, at ranks 1 and 2. Both repeat the question's words: L2 "minor physical upgrades", L3 "developers" and "performance". Meaning search ranked them 9th and 42nd, and returned none of the 4 listed sentences.

**Observation:** no search returned L1 or L4 in its top 4. L1, "Valve has announced a new Steam Deck", ranked lowest of the four listed sentences in every search. The question asks which company, so it never names Valve or the Steam Deck: L1 shares almost no words with the question, and it names its topic with words the question does not use.

**Observation:** meaning search's top 4 are other sentences about tech products, events and games, none about Valve. The question describes the product without naming it.

**Hypothesis:** because the question never names the product, its embedding points at news about new tech products in general, which many sentences in the collection match.

<u>Question 2</u>

In [10]:
side_by_side(972)

Question: Who is the individual facing a criminal trial on fraud charges, previously likened to a prominent investor, and is accused of misusing customer funds from a cryptocurrency exchange, as reported by TechCrunch and The Verge?
Answer in the dataset: Sam Bankman-Fried

Sentences listed for this question:
  L1: The highly anticipated criminal trial for Sam Bankman-Fried, former CEO of bankrupt crypto exchange FTX, started Tuesday to determine whether he’s guilty of seven counts of fraud and conspiracy.
  L2: Bankman-Fried was even compared to Warren Buffet and many called him the white horse of crypto (TechCrunch never did, for what it’s worth).
  L3: It was Bankman-Fried’s decision, she said, as he was the CEO of FTX.
  L4: The prosecution painted Bankman-Fried as someone who knowingly committed fraud to achieve great wealth, power and influence, while the defense countered that the FTX founder acted in good faith, never meant to commit fraud or steal and basically got in over his

**Observation:** meaning search returned 3 of the 4 listed sentences. L2 says Bankman-Fried "was even compared to Warren Buffet", where the question says "previously likened to a prominent investor": the same meaning in different words. Meaning search ranked L2 3rd, keyword search 25th.

**Observation:** here hybrid search returned fewer listed sentences than meaning search. L2's hybrid score = 1 / (60 + 3) + 1 / (60 + 25) = 0.02764, ranked 8th. The two unlisted sentences in the hybrid top 4 were ranked within the top 6 by both searches, so each got a higher hybrid score, 0.03102 and 0.03077. In Step 4, L1 also had keyword rank 25, but meaning rank 1, which gave it 0.02816 and rank 2.

**Conclusion:** no way is always better. Keyword search finds sentences that repeat the question's words, meaning search finds sentences that say the same thing in other words, and hybrid search found more listed sentences than both in Step 4, as many as keyword search in the first question here, and fewer than meaning search in the second.

## Key takeaways

- The vector store keeps every sentence's embedding in one table, made once. A new question is turned into an embedding when it is asked, and compared with every row.
- Meaning search returns the sentences whose embeddings are closest to the question's embedding. It finds sentences that say the same thing in other words, but can miss a sentence that repeats the question's exact words.
- Keyword search, scored with BM25, returns the sentences that share the question's rarest words. It finds exact names and phrases, but misses a sentence that uses other words, or even other forms of the same word, such as "fans" for "fan".
- Hybrid search combines the two rankings with reciprocal rank fusion. A sentence ranked high by both searches can beat one ranked first by one search alone.
- On three hand-picked questions, no way found the most listed sentences every time. Three questions cannot say which way is better on average.

## Conclusion

Meaning search and keyword search find different sentences, so each misses some listed sentences the other finds, and hybrid search, which combines them, can find both but is not always better than the better of the two.